<a href="https://colab.research.google.com/github/himanshu-gangwar/AI_Learning/blob/main/langchain_groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦜 LangChain + Groq: Multi-Agent AI Systems
## The Same Research Report Generator — Built with LangChain

---

### 🎯 What's Different Here vs the CrewAI Version?

| Concept | CrewAI Way | LangChain Way |
|---------|-----------|---------------|
| **Agent definition** | `Agent(role=, goal=, backstory=)` — declarative | `ChatPromptTemplate` + `ChatGroq` — explicit |
| **Chaining** | `context=[task1]` — automatic | `prev_output \|chain` — manual, visible |
| **Orchestration** | `Crew` + `Process.sequential` hidden | You write the pipeline yourself — nothing hidden |
| **LLM setup** | String `'groq/model'` via LiteLLM | `ChatGroq(model='...')` directly |
| **Mental model** | Team of employees | Assembly line of prompt → LLM → parser steps |

### 🏗️ What We're Building
The **exact same 3-agent Research Report Generator** from the CrewAI notebook:
1. 🔍 **Researcher** — Gathers info on a topic  
2. 🧠 **Analyst** — Finds insights from the research  
3. ✍️ **Writer** — Produces a polished final report  

But this time, **you can see every wire** — LangChain makes the plumbing explicit.

> **Why learn both?** CrewAI is faster to prototype. LangChain gives you more control and is easier to customise deeply. Knowing both makes you a stronger agentic AI developer.

---
## 📦 Step 1: Install Dependencies

LangChain is modular — you install only the parts you need:
- `langchain` — core abstractions (chains, prompts, memory)
- `langchain-groq` — the ChatGroq LLM integration
- `langchain-core` — base interfaces (usually installed as a dependency)

No LiteLLM needed here — LangChain talks directly to Groq via `langchain-groq`.

In [ ]:
!pip install langchain langchain-groq langchain-core -q

import langchain, langchain_groq
print(f"✅ langchain:      {langchain.__version__}")
print(f"✅ langchain-groq: {langchain_groq.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.1 MB/s eta 0:00:00
✅ langchain:      1.2.14
✅ langchain-groq: 1.1.2


---
## 🔑 Step 2: Configure API Key & LLM

### Key difference from CrewAI:
In CrewAI we passed a **string** (`'groq/llama-3.3-70b-versatile'`) and LiteLLM handled the rest.  
In LangChain we **directly instantiate** a `ChatGroq` object — the LLM is a first-class Python object.

```
CrewAI:     llm = "groq/llama-3.3-70b-versatile"   ← string, routed via LiteLLM
LangChain:  llm = ChatGroq(model="llama-3.3-70b-versatile")  ← object, direct API call
```

In [ ]:
import os
from getpass import getpass
from langchain_groq import ChatGroq

# Securely enter API key
groq_api_key = getpass("🔑 Enter your Groq API Key: ")
os.environ["GROQ_API_KEY"] = groq_api_key

# ─────────────────────────────────────────────────────────
# 🧠 CONCEPT: ChatGroq — LangChain's Groq integration
# ─────────────────────────────────────────────────────────
# ChatGroq is a LangChain "Chat Model" — it wraps Groq's
# API and exposes a standard .invoke() interface.
#
# All LangChain chat models share the same interface:
#   llm.invoke([HumanMessage(content="hello")])
# This means you can swap ChatGroq for ChatOpenAI,
# ChatAnthropic etc. with zero other code changes.
# THAT is LangChain's superpower.
# ─────────────────────────────────────────────────────────

llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # No 'groq/' prefix needed here!
    temperature=0.7,
    groq_api_key=groq_api_key
)

# Quick test
from langchain_core.messages import HumanMessage
test = llm.invoke([HumanMessage(content="Reply with exactly: LangChain + Groq is working!")])
print(f"✅ LLM test: {test.content}")

🔑 Enter your Groq API Key: ··········
✅ LLM test: LangChain + Groq is working!


---
## 🧠 Step 3: Understanding LangChain's Core Building Blocks

Before building agents, understand the 3 primitives we'll use:

```
┌─────────────────────────────────────────────────────────┐
│              LCEL CHAIN (LangChain Expression Language)  │
│                                                         │
│   PromptTemplate  →  ChatGroq LLM  →  StrOutputParser  │
│        │                  │                  │          │
│   Formats the         Calls Groq         Extracts the  │
│   system+user         API with           text string    │
│   prompt             the prompt          from response  │
│                                                         │
│   chain = prompt | llm | parser   ← The | pipe syntax  │
└─────────────────────────────────────────────────────────┘
```

### The | (pipe) operator
LangChain uses `|` to chain steps together. Output of the left flows as input to the right.  
This is called **LCEL** — LangChain Expression Language.  
It's similar to Unix pipes: `cat file.txt | grep 'word' | wc -l`

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ─────────────────────────────────────────────────────────
# CONCEPT: ChatPromptTemplate
# ─────────────────────────────────────────────────────────
# Defines the structure of the prompt sent to the LLM.
# - system: sets the agent's persona (like role+goal+backstory in CrewAI)
# - human:  the actual task/question with {variables} to fill in
# ─────────────────────────────────────────────────────────

# Minimal demo: a single agent call
demo_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("human", "What is {concept} in one sentence?")
])

# StrOutputParser extracts just the text string from the LLM's AIMessage response
parser = StrOutputParser()

# Build the chain using the pipe operator
demo_chain = demo_prompt | llm | parser

# Run it — .invoke() passes the variable values
result = demo_chain.invoke({"concept": "quantum entanglement"})
print("Demo chain output:")
print(result)

Demo chain output:
Quantum entanglement is a phenomenon where two or more particles become connected in such a way that their properties are correlated, regardless of the distance between them.


---
## 🔍 Step 4: Building the Researcher Agent

### How an 'Agent' Works in LangChain

In LangChain, an "agent" is really just a **chain** with a carefully crafted prompt that assigns a persona.  
The `system` message is where you put the role + backstory.  
The `human` message is where you put the task + any input variables.

```
CrewAI Agent:                    LangChain Equivalent:
  role="Researcher"         →    system: "You are a Senior Research Specialist..."
  goal="Compile research"   →    system: "Your goal is to compile research..."
  backstory="15 yrs exp"    →    system: "You have 15 years of experience..."
  task.description          →    human:  "Research this topic: {topic}"
```

In [ ]:
# ─────────────────────────────────────────────────────────
# 🔍 AGENT 1: Researcher
# ─────────────────────────────────────────────────────────
# The system message IS the agent's identity.
# We pack role + goal + backstory all into one system prompt.
# ─────────────────────────────────────────────────────────

researcher_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a Senior Research Specialist with 15 years of experience across \
technology, science, and business domains.

Your goal is to conduct thorough research on any given topic and compile \
comprehensive, accurate information.

You are known for your ability to quickly identify the most important aspects \
of any topic and present information in a structured, factual manner. \
You always cover multiple perspectives and highlight why a topic matters.

Always structure your research brief with these 6 sections:
1. Core Concepts & Fundamentals
2. Current State & Recent Developments
3. Key Players, Companies & Researchers
4. Real-World Applications & Use Cases
5. Challenges & Limitations
6. Future Outlook & Predictions"""
    ),
    (
        "human",
        """Conduct comprehensive research on this topic: **{topic}**

Compile all findings into a detailed research brief (600-800 words).
Use bullet points and short paragraphs. Be specific — include facts and numbers."""
    )
])

# Build the researcher chain: prompt → LLM → parse to string
researcher_chain = researcher_prompt | llm | StrOutputParser()

print("✅ Researcher agent chain built")
print(f"   Components: ChatPromptTemplate | ChatGroq | StrOutputParser")
print(f"   Input variables: {researcher_prompt.input_variables}")

✅ Researcher agent chain built
   Components: ChatPromptTemplate | ChatGroq | StrOutputParser
   Input variables: ['topic']


---
## 🧠 Step 5: Building the Analyst Agent

### Key Difference: Passing Context Between Agents

In CrewAI, you pass `context=[research_task]` and it injects the output automatically.  
In LangChain, **you do it yourself** — the prior agent's output becomes an input variable `{research_output}` in the next prompt.

This is more verbose, but you can see exactly what's happening.

In [ ]:
# ─────────────────────────────────────────────────────────
# 🧠 AGENT 2: Analyst
# ─────────────────────────────────────────────────────────
# Notice: the human message accepts {research_output}
# That's where Task 1's output gets injected.
# ─────────────────────────────────────────────────────────

analyst_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a Strategic Analyst with a background in consulting and data science.

Your goal is to analyse research findings, identify key patterns and insights, \
evaluate implications, and structure information into clear sections with actionable takeaways.

You excel at taking raw information and transforming it into structured insights. \
You identify trends, spot contradictions, evaluate risks and opportunities, \
and always ask 'so what?' — turning facts into meaningful conclusions.

Always structure your analysis with these 6 sections:
1. Key Insights (top 3-5 takeaways)
2. Opportunities
3. Risks & Challenges
4. Stakeholder Impact
5. Timeline Assessment (short-term 1-2yr vs long-term 5-10yr)
6. Recommended Actions"""
    ),
    (
        "human",
        """Analyse the following research on **{topic}**.

--- RESEARCH BRIEF ---
{research_output}
--- END RESEARCH BRIEF ---

Produce a structured strategic analysis (400-600 words) covering all 6 sections above."""
    )
])

analyst_chain = analyst_prompt | llm | StrOutputParser()

print("✅ Analyst agent chain built")
print(f"   Input variables: {analyst_prompt.input_variables}")
print("   ↑ Note: needs both 'topic' AND 'research_output' from Agent 1")

✅ Analyst agent chain built
   Input variables: ['research_output', 'topic']
   ↑ Note: needs both 'topic' AND 'research_output' from Agent 1


---
## ✍️ Step 6: Building the Writer Agent

The Writer receives **both** the research output and the analysis output.  
In CrewAI this was `context=[research_task, analysis_task]`.  
Here we explicitly add `{research_output}` and `{analysis_output}` as prompt variables.

In [ ]:
# ─────────────────────────────────────────────────────────
# ✍️ AGENT 3: Writer
# ─────────────────────────────────────────────────────────
# Takes BOTH prior outputs — research + analysis
# and synthesises them into a final polished report.
# ─────────────────────────────────────────────────────────

writer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an experienced Content Writer & Editor who has written hundreds of \
reports, whitepapers, and articles.

Your goal is to transform research and analysis into compelling, well-structured reports \
that are engaging, clear, and appropriate for a professional non-specialist audience.

Your writing style is clear, engaging, and authoritative. You always:
- Begin with a compelling executive summary
- Use clear section headings
- Explain technical concepts with concrete examples
- End with a forward-looking conclusion
- Format in Markdown (##, **, bullet points)"""
    ),
    (
        "human",
        """Write a comprehensive report on **{topic}** using the research and analysis below.

--- RESEARCH ---
{research_output}

--- ANALYSIS ---
{analysis_output}
--- END ---

Produce a polished Markdown report (800-1000 words) with:
- A title and executive summary (100 words)
- 4-5 clearly structured body sections
- A conclusion with forward-looking perspective"""
    )
])

writer_chain = writer_prompt | llm | StrOutputParser()

print("✅ Writer agent chain built")
print(f"   Input variables: {writer_prompt.input_variables}")
print("   ↑ Needs 'topic', 'research_output', AND 'analysis_output'")
print("\n🎉 All 3 agent chains ready!")

✅ Writer agent chain built
   Input variables: ['analysis_output', 'research_output', 'topic']
   ↑ Needs 'topic', 'research_output', AND 'analysis_output'

🎉 All 3 agent chains ready!


---
## 🔗 Step 7: Building the Pipeline (The 'Crew' Equivalent)

### The Big Conceptual Difference

```
CrewAI:                              LangChain:
──────────────────────────────       ──────────────────────────────
crew = Crew(                         def run_pipeline(topic):
  agents=[researcher,                  research = researcher_chain
          analyst,                                 .invoke({topic})
          writer],                     analysis = analyst_chain
  tasks=[t1, t2, t3],                             .invoke({topic,
  process=sequential                               research})
)                                      report   = writer_chain
crew.kickoff()                                     .invoke({topic,
                                                    research,
# CrewAI hides the pipeline           analysis})
# You declare what, it decides how    return report
                                     # YOU write the pipeline
                                     # Full control & visibility
```

Neither is better — they're different trade-offs. CrewAI: faster to build. LangChain: more control.

In [ ]:
import time

def run_research_pipeline(topic: str, verbose: bool = True) -> dict:
    """
    Runs the full 3-agent pipeline:
    Researcher → Analyst → Writer

    Returns a dict with all intermediate outputs
    so you can inspect each stage.
    """
    results = {}
    start = time.time()

    # ─── STAGE 1: Researcher ────────────────────────────
    if verbose:
        print("🔍 Stage 1/3: Researcher Agent running...")

    research_output = researcher_chain.invoke({"topic": topic})
    results["research"] = research_output

    if verbose:
        print(f"   ✅ Research complete ({len(research_output.split())} words)")
        print()

    # ─── STAGE 2: Analyst ───────────────────────────────
    # Note: manually passing research_output as context
    if verbose:
        print("🧠 Stage 2/3: Analyst Agent running...")

    analysis_output = analyst_chain.invoke({
        "topic": topic,
        "research_output": research_output   # ← inject Stage 1 output
    })
    results["analysis"] = analysis_output

    if verbose:
        print(f"   ✅ Analysis complete ({len(analysis_output.split())} words)")
        print()

    # ─── STAGE 3: Writer ────────────────────────────────
    # Note: passing BOTH prior outputs as context
    if verbose:
        print("✍️  Stage 3/3: Writer Agent running...")

    report_output = writer_chain.invoke({
        "topic": topic,
        "research_output": research_output,  # ← inject Stage 1
        "analysis_output": analysis_output   # ← inject Stage 2
    })
    results["report"] = report_output

    duration = round(time.time() - start, 1)
    results["duration"] = duration

    if verbose:
        print(f"   ✅ Report complete ({len(report_output.split())} words)")
        print()
        print("=" * 60)
        print(f"🏁 Pipeline complete in {duration}s")
        print("=" * 60)

    return results


print("✅ Pipeline function defined!")
print("   run_research_pipeline(topic) will run all 3 agents")

✅ Pipeline function defined!
   run_research_pipeline(topic) will run all 3 agents


---
## 🚀 Step 8: Run the Pipeline!

> ⏱️ Takes 30–90 seconds. Watch the stage-by-stage output.

In [ ]:
# ─── Set your topic ────────────────────────────────────────
TOPIC = "Quantum Computing and its impact on Cybersecurity"
# ───────────────────────────────────────────────────────────

print(f"📌 Topic: {TOPIC}")
print("=" * 60)
print()

# Run the full pipeline
results = run_research_pipeline(TOPIC)

📌 Topic: Quantum Computing and its impact on Cybersecurity

🔍 Stage 1/3: Researcher Agent running...
   ✅ Research complete (924 words)

🧠 Stage 2/3: Analyst Agent running...
   ✅ Analysis complete (524 words)

✍️  Stage 3/3: Writer Agent running...
   ✅ Report complete (715 words)

🏁 Pipeline complete in 7.7s


In [ ]:
# Inspect intermediate outputs — something CrewAI hides by default
print("\n" + "=" * 60)
print("📋 STAGE 1 OUTPUT — Raw Research Brief:")
print("=" * 60)
print(results["research"])


📋 STAGE 1 OUTPUT — Raw Research Brief:
**1. Core Concepts & Fundamentals**
Quantum computing is a revolutionary technology that uses the principles of quantum mechanics to perform calculations and operations on data. Unlike classical computers, which use bits to store and process information, quantum computers use quantum bits or qubits. These qubits can exist in multiple states simultaneously, allowing for exponentially faster processing of complex calculations. In the context of cybersecurity, quantum computing has the potential to significantly impact the field by breaking certain types of encryption and creating new, quantum-resistant encryption methods.

Key concepts in quantum computing and cybersecurity include:
* Quantum parallelism: the ability of qubits to perform multiple calculations simultaneously
* Quantum entanglement: the connection between qubits that allows for secure communication
* Quantum key distribution (QKD): a method of secure communication that uses quantum m

In [ ]:
print("=" * 60)
print("🧠 STAGE 2 OUTPUT — Strategic Analysis:")
print("=" * 60)
print(results["analysis"])

🧠 STAGE 2 OUTPUT — Strategic Analysis:
**1. Key Insights**
The research on quantum computing and its impact on cybersecurity reveals three key takeaways:
* Quantum computing has the potential to break certain types of encryption, but it also enables the creation of new, quantum-resistant encryption methods.
* The development of quantum-resistant cryptography is an ongoing process, with several companies and organizations, such as Google, IBM, and NIST, driving innovation in this field.
* The adoption of quantum computing in cybersecurity is expected to increase significantly in the next 5-10 years, with widespread adoption of quantum-resistant cryptography and the development of practical quantum computers.

**2. Opportunities**
The integration of quantum computing in cybersecurity presents several opportunities, including:
* Enhanced security: quantum-resistant cryptography can provide unbreakable encryption, ensuring the security of sensitive information.
* Improved simulation: quant

In [ ]:
# Render the final report as formatted Markdown
from IPython.display import Markdown, display

print("✍️  FINAL REPORT (Rendered Markdown):")
display(Markdown(results["report"]))

✍️  FINAL REPORT (Rendered Markdown):


## Quantum Computing and its Impact on Cybersecurity
### Executive Summary
Quantum computing is a revolutionary technology that uses quantum mechanics to perform calculations and operations on data, with the potential to significantly impact cybersecurity. This report explores the current state of quantum computing, its applications in cybersecurity, and the challenges and limitations associated with its adoption. As quantum computing continues to evolve, it is essential to understand its potential impact on cybersecurity and take proactive steps to prepare for the future.

## Introduction to Quantum Computing and Cybersecurity
Quantum computing is a rapidly evolving field that uses the principles of quantum mechanics to perform calculations and operations on data. Unlike classical computers, which use bits to store and process information, quantum computers use quantum bits or qubits. These qubits can exist in multiple states simultaneously, allowing for exponentially faster processing of complex calculations. In the context of cybersecurity, quantum computing has the potential to break certain types of encryption and create new, quantum-resistant encryption methods. Key concepts in quantum computing and cybersecurity include quantum parallelism, quantum entanglement, and quantum key distribution (QKD).

## Applications of Quantum Computing in Cybersecurity
Quantum computing has several real-world applications in cybersecurity, including cryptanalysis, quantum key distribution, and quantum-resistant cryptography. Quantum computers can potentially break certain types of encryption, such as RSA and elliptic curve cryptography, but they can also be used to create new, quantum-resistant encryption methods. Quantum key distribution is a method of secure communication that uses quantum mechanics to encode and decode messages. Examples of real-world applications include the use of quantum key distribution to secure communication networks, such as the Secure Communication Network (SCN) developed by the Chinese government, and the development of quantum-resistant cryptography protocols, such as the New Hope protocol developed by Google and the University of California, Los Angeles (UCLA).

## Challenges and Limitations of Quantum Computing in Cybersecurity
Despite the potential of quantum computing to revolutionize cybersecurity, there are several challenges and limitations to its adoption. These include quantum noise and error correction, scalability, and the need for education and awareness about the potential impact of quantum computing on cybersecurity. Currently, quantum computers are small-scale and limited in their processing power, which can limit their adoption in cybersecurity. Additionally, the development of standards and guidelines for quantum-resistant cryptography is an ongoing process. Statistics on the challenges and limitations of quantum computing include a recent survey by the Ponemon Institute, which found that 71% of organizations are not prepared for the potential impact of quantum computing on their cybersecurity.

## Future Outlook and Predictions
The future outlook for quantum computing and cybersecurity is rapidly evolving, with significant advancements expected in the coming years. Predictions include the widespread adoption of quantum-resistant cryptography, the development of practical quantum computers, and an increased focus on quantum cybersecurity research and development. Statistics on the future outlook and predictions include a recent report by McKinsey, which estimated that the quantum computing market will reach $1 billion by 2025, and a survey by the Quantum Computing Report, which found that 60% of respondents expect quantum computing to have a significant impact on cybersecurity within the next 5 years. To prepare for the future, organizations should start investing in quantum-resistant cryptography and quantum computing research and development, governments should develop and implement standards and guidelines for quantum-resistant cryptography, and individuals should educate themselves about the potential impact of quantum computing on their personal data and security.

## Conclusion
In conclusion, quantum computing has the potential to significantly impact cybersecurity, with both positive and negative consequences. As quantum computing continues to evolve, it is essential to understand its potential impact on cybersecurity and take proactive steps to prepare for the future. This includes investing in quantum-resistant cryptography, developing standards and guidelines for quantum-resistant cryptography, and educating individuals about the potential impact of quantum computing on their personal data and security. With the widespread adoption of quantum-resistant cryptography and the development of practical quantum computers, we can expect to see significant advancements in cybersecurity in the coming years. Ultimately, the future of cybersecurity will depend on our ability to harness the power of quantum computing while mitigating its risks, and it is essential that we take a proactive and forward-looking approach to prepare for the challenges and opportunities that lie ahead.

In [ ]:
# Save all outputs to files
safe_topic = TOPIC.replace(" ", "_").replace("/", "-")

with open(f"{safe_topic}_research.md", "w") as f:
    f.write(f"# Research Brief: {TOPIC}\n\n" + results["research"])

with open(f"{safe_topic}_analysis.md", "w") as f:
    f.write(f"# Analysis: {TOPIC}\n\n" + results["analysis"])

with open(f"{safe_topic}_report.md", "w") as f:
    f.write(results["report"])

print(f"✅ Saved 3 files:")
print(f"   {safe_topic}_research.md")
print(f"   {safe_topic}_analysis.md")
print(f"   {safe_topic}_report.md")

✅ Saved 3 files:
   Quantum_Computing_and_its_impact_on_Cybersecurity_research.md
   Quantum_Computing_and_its_impact_on_Cybersecurity_analysis.md
   Quantum_Computing_and_its_impact_on_Cybersecurity_report.md


---
## ⚡ Step 9: BONUS — Streaming Output

One of LangChain's advantages: **streaming** is built in.  
Instead of waiting for the full response, you see tokens as they arrive — great for UX.

CrewAI can stream too, but it requires more setup. In LangChain it's just `.stream()` instead of `.invoke()`.

In [ ]:
# ─────────────────────────────────────────────────────────
# BONUS: Streaming with LangChain
# .stream() yields tokens as they arrive from Groq
# ─────────────────────────────────────────────────────────

print("⚡ Streaming the Researcher output in real-time:\n")
print("-" * 50)

for chunk in researcher_chain.stream({"topic": "Large Language Models"}):
    print(chunk, end="", flush=True)  # Print each token as it arrives

print("\n" + "-" * 50)
print("\n✅ Streaming complete!")

---
## 📊 Step 10: CrewAI vs LangChain — Side-by-Side Comparison

Now that you've built the same thing both ways, here's the full comparison:

In [ ]:
comparison = """
╔══════════════════════╦═════════════════════════════╦════════════════════════════════╗
║ Feature              ║ CrewAI                      ║ LangChain (LCEL)               ║
╠══════════════════════╬═════════════════════════════╬════════════════════════════════╣
║ Agent definition     ║ Agent(role=, goal=,         ║ ChatPromptTemplate             ║
║                      ║   backstory=)               ║   with system message          ║
╠══════════════════════╬═════════════════════════════╬════════════════════════════════╣
║ LLM setup            ║ String: 'groq/model-name'   ║ ChatGroq(model='model-name')  ║
║                      ║ via LiteLLM                 ║ direct API object              ║
╠══════════════════════╬═════════════════════════════╬════════════════════════════════╣
║ Agent chaining       ║ context=[task1]             ║ Manual: pass output as         ║
║                      ║ automatic injection         ║ {research_output} variable     ║
╠══════════════════════╬═════════════════════════════╬════════════════════════════════╣
║ Orchestration        ║ Crew + Process.sequential   ║ You write the Python function  ║
╠══════════════════════╬═════════════════════════════╬════════════════════════════════╣
║ Streaming            ║ Needs extra config          ║ .stream() built-in             ║
╠══════════════════════╬═════════════════════════════╬════════════════════════════════╣
║ Memory               ║ memory=True (w/ config)     ║ ConversationBufferMemory       ║
╠══════════════════════╬═════════════════════════════╬════════════════════════════════╣
║ Tools                ║ crewai-tools prebuilt       ║ @tool decorator or LangChain   ║
║                      ║                             ║ community tools                ║
╠══════════════════════╬═════════════════════════════╬════════════════════════════════╣
║ Visibility           ║ Hidden plumbing             ║ You see every wire             ║
╠══════════════════════╬═════════════════════════════╬════════════════════════════════╣
║ Lines of code        ║ ~30 lines for 3 agents      ║ ~80 lines for 3 agents         ║
╠══════════════════════╬═════════════════════════════╬════════════════════════════════╣
║ Best for             ║ Quick prototypes,           ║ Custom workflows,              ║
║                      ║ role-based pipelines        ║ production systems,            ║
║                      ║                             ║ teaching how LLMs work         ║
╚══════════════════════╩═════════════════════════════╩════════════════════════════════╝
"""
print(comparison)

---
## 🧩 Step 11: Core Concepts Summary

| LangChain Primitive | What it does | Code |
|---------------------|-------------|------|
| `ChatGroq` | Wraps Groq's API — the LLM object | `ChatGroq(model='llama-3.3-70b-versatile')` |
| `ChatPromptTemplate` | Defines system + human messages with variables | `from_messages([("system","..."), ("human","{topic}")])` |
| `StrOutputParser` | Extracts plain text string from LLM response | `StrOutputParser()` |
| LCEL `\|` pipe | Chains steps: output flows left to right | `prompt \| llm \| parser` |
| `.invoke()` | Run the chain with input variables | `chain.invoke({"topic": "AI"})` |
| `.stream()` | Stream tokens as they arrive | `for chunk in chain.stream({...})` |

### 🔑 The Golden Rule
> **In LangChain, you are responsible for passing context between agents.**  
> Take Stage 1's output → put it in Stage 2's input variables → Stage 2 sees it.  
> This is explicit, transparent, and completely within your control.

### 🚀 What to Build Next
- Add **memory** with `ConversationBufferMemory` so agents remember past runs
- Add **tools** with `@tool` decorator — give your researcher a real web search
- Use **LangGraph** for non-linear workflows (branching, loops, parallel agents)
- Add **LangSmith** for observability — trace every LLM call visually

---
> **Resources:**  
> - [LangChain Docs](https://python.langchain.com/docs)  
> - [Groq + LangChain Guide](https://console.groq.com/docs/langchain)  
> - [LCEL Explained](https://python.langchain.com/docs/expression_language)  
> - [LangGraph (advanced)](https://langchain-ai.github.io/langgraph)

In [ ]:
# ─────────────────────────────────────────────────────────
# 🎓 QUICK QUIZ — Test your understanding!
# ─────────────────────────────────────────────────────────

quiz = [
    ("Q1", "What LangChain class wraps Groq's API?",
     "ChatGroq from langchain_groq"),

    ("Q2", "What does the | (pipe) operator do in LangChain LCEL?",
     "Chains steps together — output of left becomes input of right"),

    ("Q3", "How do you pass Agent 1's output to Agent 2 in LangChain?",
     "Manually — store Agent 1's output, pass it as a variable in Agent 2's prompt"),

    ("Q4", "What's the LangChain equivalent of CrewAI's context=[task1]?",
     "Adding {research_output} as a variable in the next prompt template"),

    ("Q5", "How do you stream token-by-token output in LangChain?",
     "Use .stream() instead of .invoke() — e.g. for chunk in chain.stream({...})"),

    ("Q6", "What does StrOutputParser do?",
     "Extracts the plain text string from the LLM's AIMessage response object"),
]

print("🎓 QUICK QUIZ — Cover the answers and test yourself!")
print("=" * 60)
for q_id, question, answer in quiz:
    print(f"\n{q_id}: {question}")
    print(f"   💡 {answer}")